# Design an agent + evaluation harness

**Agents & RL** · harness design — the scaffolding every agent project needs.

We build the reusable pieces: a **tool** registry, a **reasoning loop** that picks and calls tools, a **task suite**, and an **evaluator** that runs the agent over the suite and scores it. The agent here is rule-based so it is self-contained and verifiable — swap in an LLM policy and the harness is unchanged.

> CPU is fine. No LLM/network needed.

In [ ]:
import os, re, json, math, matplotlib.pyplot as plt

## 1 · Tools (a registry the agent can call)

In [ ]:
TOOLS = {
    "calc": lambda expr: eval(expr, {"__builtins__": {}}, {"sqrt": math.sqrt}),
    "len":  lambda s: len(s),
    "rev":  lambda s: s[::-1],
}

## 2 · A tool-using agent (a ReAct-style loop)
Parse the task, decide which tool to call, return the answer. (Replace this function body with an LLM call and the rest of the harness is identical.)

In [ ]:
def agent(task):
    log = []
    m = re.match(r"compute (.+)", task)
    if m: tool, arg = "calc", m.group(1)
    elif task.startswith("length of "): tool, arg = "len", task[len("length of "):]
    elif task.startswith("reverse "): tool, arg = "rev", task[len("reverse "):]
    else: return None, [("noop", task)]
    log.append(("call", tool, arg)); out = TOOLS[tool](arg); log.append(("obs", out))
    return out, log

## 3 · A task suite with ground-truth answers

In [ ]:
SUITE = [
    {"task": "compute 2*(3+4)", "answer": 14},
    {"task": "compute sqrt(144)", "answer": 12.0},
    {"task": "length of robotics", "answer": 8},
    {"task": "reverse agent", "answer": "tnega"},
    {"task": "compute 10*10-1", "answer": 99},
]

## 4 · The evaluator — run the agent over the suite and score

In [ ]:
def evaluate(agent_fn, suite):
    results = []
    for t in suite:
        try: pred, log = agent_fn(t["task"])
        except Exception as e: pred, log = f"error:{e}", []
        ok = (pred == t["answer"])
        results.append({"task": t["task"], "pred": pred, "gold": t["answer"], "ok": bool(ok), "trace": log})
    acc = sum(r["ok"] for r in results) / len(results)
    return acc, results
acc, results = evaluate(agent, SUITE)
print(f"success rate: {acc:.2f}\n")
for r in results: print(("PASS" if r["ok"] else "FAIL"), "|", r["task"], "->", r["pred"])

## 5 · Report

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh([r["task"] for r in results], [1 if r["ok"] else 0 for r in results], color=["#5aa86a" if r["ok"] else "#e0796b" for r in results])
ax.set_xlim(0, 1); ax.set_title(f"agent harness — {acc:.0%} pass"); ax.set_xticks([])
plt.tight_layout(); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/AG_agent_harness/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/AG_agent_harness"; os.makedirs(run, exist_ok=True)
json.dump({"success_rate": acc, "results": [{"task": r["task"], "pred": str(r["pred"]), "ok": r["ok"]} for r in results]}, open(f"{run}/results.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
The harness scores **LM** agents on **C** egocentric-assistant and **D** embodied tasks.

### Where to go next
- Replace `agent()` with an **LLM** that emits tool calls (function calling / ReAct) — see the advanced *LLM agent* lab.
- Add multi-step planning, memory, and a larger graded suite; log full traces for failure analysis (lesson C9).